- geo3k multi-turn
    - https://github.com/THUDM/slime/tree/main/examples/geo3k_vlm_multi_turn
    - 由 prompt 约定、XML/JSON parser 和 Python env 共同实现的内嵌工具协议。
        - https://huggingface.co/datasets/VeraIsHere/geo3k_imgurl_processed
    - tool feedback
        - https://github.com/THUDM/slime/blob/main/examples/geo3k_vlm_multi_turn/env_geo3k.py#L150

### slime 的训练数据契约（training contract）

- 明确的 token-in / token-out
    - https://github.com/THUDM/slime/blob/main/examples/retool/generate_with_retool.py

### 一个 async generate 就是 agent

```
generate_rollout()
  └─ generate_rollout_async()
       ├─ DataSource 为每个 prompt 复制 8 个 Sample
       ├─ submit_generate_tasks()
       └─ generate_and_rm_group()
            ├─ asyncio.create_task(generate_and_rm(sample_1))
            ├─ asyncio.create_task(generate_and_rm(sample_2))
            └─ ...
                 └─ generate_and_rm()
                      ├─ await generate_with_search.generate(...)
                      └─ await reward_func(...)
```

- 以 search-r1 为例：examples/search-r1/generate_with_search.py:145 的函数

```python
async def generate(args, sample: Sample, sampling_params) -> Sample:

    for _turn_idx in range(SEARCH_R1_CONFIGS["max_turns"]):
        output = await post(sglang_url, payload)
        cur_response = output["text"]
    
        next_obs, done = await execute_predictions(cur_response)
        if done:
            break
    
        response += next_obs
```